# 184. LATS：怎样用 MCTS 统一 Agent 的推理、行动与规划？

> **面试问题：Selection、Expansion、Simulation、Backpropagation 如何落到工具环境？为什么不能在真实副作用上随意分支？**

## 先给结论

LATS 把语言模型提案、环境反馈、价值评估和反思放进 MCTS。每个节点必须对应可恢复的环境 state；UCT/PUCT 在已知高价值和未探索动作间权衡。搜索分支只应运行在模拟器、快照或只读工具中，最终选定路径才提交真实副作用。

## 推荐回答主线

1. 定义可哈希 state、合法 action、确定/随机 transition、终态 reward 和环境 snapshot。
2. 实现 PUCT selection、policy-prior expansion、rollout/value 与路径 backprop，检查访问次数守恒。
3. 加入 cycle/transposition、失败 reflection、深度/节点/tool/token 预算与不可行动作剪枝。
4. 只 commit 选中计划；比较 ReAct/ToT baseline 的 success、环境步数、成本和搜索方差。

## 教学实现边界

使用可逆整数环境而非真实网页/文件；启发式 prior/value 代替 LLM。它展示 MCTS 状态语义，不证明 LATS 在开放任务上的收益。

## 一手资料

- [Language Agent Tree Search](https://arxiv.org/abs/2310.04406)
- [ReAct](https://arxiv.org/abs/2210.03629)
- [AlphaZero / MCTS](https://arxiv.org/abs/1712.01815)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass, field  # 导入本单元需要的依赖。

import numpy as np  # 导入本单元需要的依赖。

# 可逆玩具环境：从 1 通过 +1/+2/*2 到 7，越界或超步失败。
ACTIONS = ("+1", "+2", "*2")  # 计算并保存当前步骤的中间状态。
TARGET, MAX_VALUE, MAX_STEPS = 7, 12, 5  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class State:  # 定义承载本节状态与行为的数据结构。
    value: int  # 执行当前语句以推进本节示例。
    steps: int  # 执行当前语句以推进本节示例。

assert TARGET < MAX_VALUE  # 用受控断言验证关键不变量。
assert len(ACTIONS) == 3  # 用受控断言验证关键不变量。
assert State(1, 0) != State(1, 1)  # 用受控断言验证关键不变量。


## 1. 环境 transition：终态由 state 推导，且不可再次执行

`done` 不能由调用者随手塞进节点，否则同一 state 可能同时被当成运行态和终态。这里用纯函数从 `(value, steps)` 推导 success、overflow 与 step budget；`transition` 在执行前先拒绝所有终态，保证终止语义只有一个事实来源。


In [ ]:
def state_key(state):  # 定义本节可复用的核心函数。
    return (state.value, state.steps)  # 返回当前分支计算出的结果。

def terminal_outcome(state):  # 定义本节可复用的核心函数。
    if state.value == TARGET:  # 按当前条件选择后续控制路径。
        return True, 1.0, "success"  # 返回当前分支计算出的结果。
    if state.value > MAX_VALUE:  # 按当前条件选择后续控制路径。
        return True, -1.0, "overflow"  # 返回当前分支计算出的结果。
    if state.steps >= MAX_STEPS:  # 按当前条件选择后续控制路径。
        return True, -1.0, "step_budget"  # 返回当前分支计算出的结果。
    return False, 0.0, "running"  # 返回当前分支计算出的结果。

def transition(state, action):  # 定义本节可复用的核心函数。
    terminal, _, reason = terminal_outcome(state)  # 计算并保存当前步骤的中间状态。
    if terminal:  # 按当前条件选择后续控制路径。
        raise RuntimeError(f"终态不可继续 transition: {reason}")  # 遇到非法合同立即显式失败。
    if action not in ACTIONS:  # 按当前条件选择后续控制路径。
        raise ValueError("非法动作")  # 遇到非法合同立即显式失败。
    value = state.value + 1 if action == "+1" else state.value + 2 if action == "+2" else state.value * 2  # 计算并保存当前步骤的中间状态。
    next_state = State(value, state.steps + 1)  # 计算并保存当前步骤的中间状态。
    done, reward, _ = terminal_outcome(next_state)  # 计算并保存当前步骤的中间状态。
    return next_state, reward, done  # 返回当前分支计算出的结果。

# 转移保持旧 state 不变；success、overflow 与耗尽步数完全由新 state 推导。
start = State(1, 0)  # 计算并保存当前步骤的中间状态。
next_state, reward, done = transition(start, "*2")  # 计算并保存当前步骤的中间状态。
assert start == State(1, 0) and next_state == State(2, 1)  # 用受控断言验证关键不变量。
assert transition(State(6, 2), "+1")[1:] == (1.0, True)  # 用受控断言验证关键不变量。
assert transition(State(11, 1), "*2")[1:] == (-1.0, True)  # 用受控断言验证关键不变量。
assert transition(State(5, MAX_STEPS - 1), "+1")[1:] == (-1.0, True)  # 用受控断言验证关键不变量。

# 反例：无论成功、越界还是耗尽步数，终态都不可继续产生后继。
blocked_terminal_transitions = 0  # 计算并保存当前步骤的中间状态。
for terminal_state in (State(TARGET, 2), State(MAX_VALUE + 1, 1), State(6, MAX_STEPS)):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        transition(terminal_state, "+1")  # 执行当前语句以推进本节示例。
    except RuntimeError:  # 捕获预期异常并验证失败分支。
        blocked_terminal_transitions += 1  # 计算并保存当前步骤的中间状态。
assert blocked_terminal_transitions == 3  # 用受控断言验证关键不变量。


## 2. Node、共享统计与 PUCT：边属于树，统计属于 state

不同 action 或父节点可能到达相同 `(value, steps)`。Node 仍保存 parent/action 以还原路径，但 `visits/value_sum` 通过 `SharedStats` 引用 transposition table；terminal/reward 则每次从不可变 state 推导。PUCT 的探索常数必须是有限正数。


In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class SharedStats:  # 定义承载本节状态与行为的数据结构。
    visits: int = 0  # 计算并保存当前步骤的中间状态。
    value_sum: float = 0.0  # 计算并保存当前步骤的中间状态。

    @property  # 为下方定义附加声明式配置。
    def q(self):  # 定义本节可复用的核心函数。
        return self.value_sum / self.visits if self.visits else 0.0  # 返回当前分支计算出的结果。

@dataclass  # 为下方定义附加声明式配置。
class Node:  # 定义承载本节状态与行为的数据结构。
    state: State  # 执行当前语句以推进本节示例。
    parent: object = None  # 计算并保存当前步骤的中间状态。
    action: str | None = None  # 计算并保存当前步骤的中间状态。
    prior: float = 1.0  # 计算并保存当前步骤的中间状态。
    stats: SharedStats = field(default_factory=SharedStats)  # 计算并保存当前步骤的中间状态。
    children: dict = field(default_factory=dict)  # 计算并保存当前步骤的中间状态。

    @property  # 为下方定义附加声明式配置。
    def terminal(self):  # 定义本节可复用的核心函数。
        return terminal_outcome(self.state)[0]  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def reward(self):  # 定义本节可复用的核心函数。
        return terminal_outcome(self.state)[1]  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def visits(self):  # 定义本节可复用的核心函数。
        return self.stats.visits  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def q(self):  # 定义本节可复用的核心函数。
        return self.stats.q  # 返回当前分支计算出的结果。

def validate_exploration(c):  # 定义本节可复用的核心函数。
    if isinstance(c, bool) or not isinstance(c, (int, float)) or not math.isfinite(c) or c <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("PUCT exploration c 必须为有限正数")  # 遇到非法合同立即显式失败。

def puct(parent, child, c=1.5):  # 定义本节可复用的核心函数。
    validate_exploration(c)  # 执行当前语句以推进本节示例。
    return child.q + c * child.prior * math.sqrt(max(parent.visits, 1)) / (1 + child.visits)  # 返回当前分支计算出的结果。

# 两条边引用同一统计对象时，一侧更新会立即反映到另一侧的 q/visits。
root = Node(start)  # 计算并保存当前步骤的中间状态。
shared = SharedStats()  # 计算并保存当前步骤的中间状态。
child_a = Node(next_state, root, "*2", prior=0.7, stats=shared)  # 计算并保存当前步骤的中间状态。
child_b = Node(State(2, 1), root, "+1", prior=0.2, stats=shared)  # 计算并保存当前步骤的中间状态。
assert child_a.stats is child_b.stats and child_a.q == 0  # 用受控断言验证关键不变量。
shared.visits, shared.value_sum = 2, 1.0  # 计算并保存当前步骤的中间状态。
assert child_a.q == child_b.q == 0.5 and child_b.visits == 2  # 用受控断言验证关键不变量。
assert puct(root, Node(State(3, 1), root, prior=0.8)) > puct(root, Node(State(3, 1), root, prior=0.1))  # 用受控断言验证关键不变量。

invalid_c_rejected = 0  # 计算并保存当前步骤的中间状态。
for invalid_c in (0, -1.0, float("nan"), float("inf"), True):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        puct(root, child_a, invalid_c)  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        invalid_c_rejected += 1  # 计算并保存当前步骤的中间状态。
assert invalid_c_rejected == 5  # 用受控断言验证关键不变量。


## 3. Expansion：policy prior、transposition table 与 forbidden 同时生效

Expansion 只为当前仍允许的 action 建边；每个后继先用 `state_key(value, steps)` 查询共享统计。这样两条同状态边不是“数值碰巧相等”，而是持有同一个统计对象；反思产生的 forbidden 也能阻止失败边重新进入树。


In [ ]:
def heuristic_priors(state, forbidden=frozenset()):  # 定义本节可复用的核心函数。
    if terminal_outcome(state)[0]:  # 按当前条件选择后续控制路径。
        return {}  # 返回当前分支计算出的结果。
    candidates = [action for action in ACTIONS if (state_key(state), action) not in forbidden]  # 计算并保存当前步骤的中间状态。
    if not candidates:  # 按当前条件选择后续控制路径。
        return {}  # 返回当前分支计算出的结果。
    distances = []  # 计算并保存当前步骤的中间状态。
    for action in candidates:  # 遍历输入元素以累积或检查结果。
        candidate, _, _ = transition(state, action)  # 计算并保存当前步骤的中间状态。
        distances.append(abs(TARGET - candidate.value))  # 计算并保存当前步骤的中间状态。
    logits = np.array([-distance for distance in distances], dtype=float)  # 计算并保存当前步骤的中间状态。
    probabilities = np.exp(logits - logits.max()); probabilities /= probabilities.sum()  # 计算并保存当前步骤的中间状态。
    return dict(zip(candidates, probabilities))  # 返回当前分支计算出的结果。

def expand(node, transpositions, forbidden=frozenset()):  # 定义本节可复用的核心函数。
    if node.terminal or node.children:  # 按当前条件选择后续控制路径。
        return  # 返回当前分支计算出的结果。
    transpositions.setdefault(state_key(node.state), node.stats)  # 计算并保存当前步骤的中间状态。
    for action, prior in heuristic_priors(node.state, forbidden).items():  # 遍历输入元素以累积或检查结果。
        state, _, _ = transition(node.state, action)  # 计算并保存当前步骤的中间状态。
        stats = transpositions.setdefault(state_key(state), SharedStats())  # 计算并保存当前步骤的中间状态。
        node.children[action] = Node(state, node, action, float(prior), stats)  # 计算并保存当前步骤的中间状态。

# +1 与 *2 都从 (1,0) 到 (2,1)，因此真实共享 table 中同一个统计对象。
root = Node(start)  # 计算并保存当前步骤的中间状态。
transpositions = {state_key(start): root.stats}  # 计算并保存当前步骤的中间状态。
expand(root, transpositions)  # 执行当前语句以推进本节示例。
assert set(root.children) == set(ACTIONS)  # 用受控断言验证关键不变量。
assert math.isclose(sum(child.prior for child in root.children.values()), 1.0)  # 用受控断言验证关键不变量。
assert all(child.parent is root for child in root.children.values())  # 用受控断言验证关键不变量。
assert root.children["+1"].state == root.children["*2"].state == State(2, 1)  # 用受控断言验证关键不变量。
assert root.children["+1"].stats is root.children["*2"].stats is transpositions[(2, 1)]  # 用受控断言验证关键不变量。

terminal_node = Node(State(TARGET, 2))  # 计算并保存当前步骤的中间状态。
expand(terminal_node, {state_key(terminal_node.state): terminal_node.stats})  # 执行当前语句以推进本节示例。
assert terminal_node.terminal and terminal_node.children == {}  # 用受控断言验证关键不变量。


## 4. Selection、reflection 与 value：失败约束进入下一次选择

Selection 每一层都过滤 `(state_key(parent), action)` forbidden，再在剩余边上比较 PUCT。Reflection 只把真实终态失败转成结构化约束；终态 value 始终使用环境 reward，不能被距离启发式覆盖。


In [ ]:
def reflection_from_failure(state, action, next_state):  # 定义本节可复用的核心函数。
    done, reward, reason = terminal_outcome(next_state)  # 计算并保存当前步骤的中间状态。
    if not done or reward >= 0:  # 按当前条件选择后续控制路径。
        return {"forbid": None, "reason": "not_failure"}  # 返回当前分支计算出的结果。
    return {"forbid": (state_key(state), action), "reason": reason}  # 返回当前分支计算出的结果。

def selectable_children(node, forbidden):  # 定义本节可复用的核心函数。
    return [child for action, child in node.children.items() if (state_key(node.state), action) not in forbidden]  # 返回当前分支计算出的结果。

def select_leaf(root, c=1.5, forbidden=frozenset()):  # 定义本节可复用的核心函数。
    validate_exploration(c)  # 执行当前语句以推进本节示例。
    node, path = root, [root]  # 计算并保存当前步骤的中间状态。
    while node.children and not node.terminal:  # 在终止条件满足前持续推进状态。
        candidates = selectable_children(node, forbidden)  # 计算并保存当前步骤的中间状态。
        if not candidates:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        node = max(candidates, key=lambda child: (puct(path[-1], child, c), child.action))  # 计算并保存当前步骤的中间状态。
        path.append(node)  # 计算并保存当前步骤的中间状态。
    return node, path  # 返回当前分支计算出的结果。

def leaf_value(node):  # 定义本节可复用的核心函数。
    if node.terminal:  # 按当前条件选择后续控制路径。
        return node.reward  # 返回当前分支计算出的结果。
    return 1.0 - min(abs(TARGET - node.state.value) / TARGET, 1.0)  # 返回当前分支计算出的结果。

# selection 返回连续路径；terminal/reward 均由 state 推导，调用者无法伪造。
leaf, path = select_leaf(root)  # 计算并保存当前步骤的中间状态。
assert path[0] is root and path[-1] is leaf  # 用受控断言验证关键不变量。
assert -1 <= leaf_value(leaf) <= 1  # 用受控断言验证关键不变量。
assert leaf_value(Node(State(TARGET, 2))) == 1.0  # 用受控断言验证关键不变量。
assert leaf_value(Node(State(MAX_VALUE + 1, 2))) == -1.0  # 用受控断言验证关键不变量。


## 5. Backpropagation：一条 simulation 给路径每个节点加一次访问

环境是单 Agent 同目标，回传同一 return；对抗游戏才交替符号。访问次数和值必须成对更新，取消/异常 simulation 不得半更新。


In [ ]:
def backpropagate(path, value):  # 定义本节可复用的核心函数。
    if not math.isfinite(value):  # 按当前条件选择后续控制路径。
        raise ValueError("回传 value 必须有限")  # 遇到非法合同立即显式失败。
    # transposition 或 cycle 可能让同一 stats 在路径出现多次；每次 simulation 只更新一次。
    updated = set()  # 计算并保存当前步骤的中间状态。
    for node in path:  # 遍历输入元素以累积或检查结果。
        identity = id(node.stats)  # 计算并保存当前步骤的中间状态。
        if identity in updated:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        node.stats.visits += 1  # 计算并保存当前步骤的中间状态。
        node.stats.value_sum += value  # 计算并保存当前步骤的中间状态。
        updated.add(identity)  # 计算并保存当前步骤的中间状态。

# 一次回传使路径上的不同共享统计各 +1，真正非路径统计保持不变。
unique_path_stats = {id(node.stats): node.stats.visits for node in path}  # 计算并保存当前步骤的中间状态。
untouched = next(child for child in root.children.values() if child.stats is not path[-1].stats)  # 计算并保存当前步骤的中间状态。
untouched_before = untouched.visits  # 计算并保存当前步骤的中间状态。
backpropagate(path, leaf_value(leaf))  # 执行当前语句以推进本节示例。
assert all(stats.visits == unique_path_stats[id(stats)] + 1 for stats in {id(node.stats): node.stats for node in path}.values())  # 用受控断言验证关键不变量。
assert untouched.visits == untouched_before  # 用受控断言验证关键不变量。
assert all(math.isfinite(node.q) for node in path)  # 用受控断言验证关键不变量。

# 同一 stats 通过两个 alias 出现在一条路径时仍只增加一次，而不是双计数。
alias = Node(path[-1].state, action="alias", stats=path[-1].stats)  # 计算并保存当前步骤的中间状态。
alias_before = alias.visits  # 计算并保存当前步骤的中间状态。
backpropagate([path[-1], alias], 0.25)  # 执行当前语句以推进本节示例。
assert alias.visits == alias_before + 1 and path[-1].visits == alias.visits  # 用受控断言验证关键不变量。


## 6. 有界 LATS 搜索：参数 fail-closed，最终动作按 root visits/Q

每次 simulation 执行 selection、带 transposition/forbidden 的 expansion、评估与原子回传。失败 reflection 立即加入集合并影响之后的 selection。搜索结束不从“任意成功轨迹”挑计划，而是按 root child 的 visit count、Q、prior 依次决策。


In [ ]:
def validate_search_config(simulations, c):  # 定义本节可复用的核心函数。
    if type(simulations) is not int or simulations <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("simulations 必须为正整数")  # 遇到非法合同立即显式失败。
    validate_exploration(c)  # 执行当前语句以推进本节示例。

def run_search(root_state, simulations=80, c=1.5):  # 定义本节可复用的核心函数。
    validate_search_config(simulations, c)  # 执行当前语句以推进本节示例。
    root = Node(root_state)  # 计算并保存当前步骤的中间状态。
    transpositions = {state_key(root_state): root.stats}  # 计算并保存当前步骤的中间状态。
    forbidden, reflections = set(), []  # 计算并保存当前步骤的中间状态。
    for _ in range(simulations):  # 遍历输入元素以累积或检查结果。
        leaf, path = select_leaf(root, c, forbidden)  # 计算并保存当前步骤的中间状态。
        if not leaf.terminal:  # 按当前条件选择后续控制路径。
            expand(leaf, transpositions, forbidden)  # 执行当前语句以推进本节示例。
            candidates = selectable_children(leaf, forbidden)  # 计算并保存当前步骤的中间状态。
            if candidates:  # 按当前条件选择后续控制路径。
                successful = [child for child in candidates if child.terminal and child.reward > 0]  # 计算并保存当前步骤的中间状态。
                chosen = max(successful or candidates, key=lambda child: (child.prior, child.action))  # 计算并保存当前步骤的中间状态。
                path.append(chosen); leaf = chosen  # 计算并保存当前步骤的中间状态。
        value = leaf_value(leaf)  # 计算并保存当前步骤的中间状态。
        backpropagate(path, value)  # 执行当前语句以推进本节示例。
        if leaf.terminal and leaf.reward < 0 and leaf.parent is not None:  # 按当前条件选择后续控制路径。
            reflection = reflection_from_failure(leaf.parent.state, leaf.action, leaf.state)  # 计算并保存当前步骤的中间状态。
            if reflection["forbid"] is not None:  # 按当前条件选择后续控制路径。
                forbidden.add(reflection["forbid"]); reflections.append(reflection)  # 计算并保存当前步骤的中间状态。
    return root, transpositions, forbidden, reflections  # 返回当前分支计算出的结果。

def choose_root_action(root, forbidden=frozenset()):  # 定义本节可复用的核心函数。
    if root.terminal:  # 按当前条件选择后续控制路径。
        raise RuntimeError("终态没有可提交 root action")  # 遇到非法合同立即显式失败。
    candidates = selectable_children(root, forbidden)  # 计算并保存当前步骤的中间状态。
    if not candidates:  # 按当前条件选择后续控制路径。
        raise RuntimeError("root 没有获准动作")  # 遇到非法合同立即显式失败。
    chosen = max(candidates, key=lambda child: (child.visits, child.q, child.prior, child.action))  # 计算并保存当前步骤的中间状态。
    return chosen.action, chosen  # 返回当前分支计算出的结果。

# 固定预算严格执行 80 次 root 更新；最终动作等于显式 visits/Q 排序结果。
search_root, search_table, learned_forbidden, reflections = run_search(start, simulations=80, c=1.5)  # 计算并保存当前步骤的中间状态。
chosen_action, chosen_child = choose_root_action(search_root, learned_forbidden)  # 计算并保存当前步骤的中间状态。
manual_choice = max(selectable_children(search_root, learned_forbidden), key=lambda child: (child.visits, child.q, child.prior, child.action))  # 计算并保存当前步骤的中间状态。
assert search_root.visits == 80  # 用受控断言验证关键不变量。
assert chosen_child is manual_choice and chosen_action == manual_choice.action  # 用受控断言验证关键不变量。
assert state_key(chosen_child.state) in search_table  # 用受控断言验证关键不变量。

invalid_search_config = 0  # 计算并保存当前步骤的中间状态。
for bad_simulations, bad_c in ((0, 1.5), (-1, 1.5), (True, 1.5), (5, 0), (5, float("nan"))):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        run_search(start, bad_simulations, bad_c)  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        invalid_search_config += 1  # 计算并保存当前步骤的中间状态。
assert invalid_search_config == 5  # 用受控断言验证关键不变量。


## 7. Transposition 与 reflection 回归：共享和禁选都要可观察

仅计算相同 key 不算复用；两个 Node 必须持有同一 `SharedStats`。同样，仅生成 reflection 文本也不算约束；把失败边设成最高 Q，加入 forbidden 后再次 selection 必须不再经过该边。


In [ ]:
# transposition 的一侧回传会改变另一 alias 的 visits/Q，且 steps 不同绝不合并。
left_alias, right_alias = root.children["+1"], root.children["*2"]  # 计算并保存当前步骤的中间状态。
right_before = right_alias.visits  # 计算并保存当前步骤的中间状态。
backpropagate([left_alias], 0.75)  # 执行当前语句以推进本节示例。
assert left_alias.stats is right_alias.stats  # 用受控断言验证关键不变量。
assert right_alias.visits == right_before + 1 and right_alias.q == left_alias.q  # 用受控断言验证关键不变量。
assert state_key(State(4, 1)) != state_key(State(4, 3))  # 用受控断言验证关键不变量。

# 构造一个故意高 Q 的 overflow 边；reflection 加入前会选它，加入后下一次 selection 必须绕开。
reflection_root = Node(State(8, 1))  # 计算并保存当前步骤的中间状态。
reflection_table = {state_key(reflection_root.state): reflection_root.stats}  # 计算并保存当前步骤的中间状态。
expand(reflection_root, reflection_table)  # 执行当前语句以推进本节示例。
reflection_root.stats.visits = 10  # 计算并保存当前步骤的中间状态。
overflow_child = reflection_root.children["*2"]  # 计算并保存当前步骤的中间状态。
overflow_child.stats.visits, overflow_child.stats.value_sum = 1, 2.0  # 计算并保存当前步骤的中间状态。
_, before_forbid_path = select_leaf(reflection_root, c=0.01, forbidden=set())  # 计算并保存当前步骤的中间状态。
reflection = reflection_from_failure(reflection_root.state, "*2", overflow_child.state)  # 计算并保存当前步骤的中间状态。
forbidden_probe = {reflection["forbid"]}  # 计算并保存当前步骤的中间状态。
_, after_forbid_path = select_leaf(reflection_root, c=0.01, forbidden=forbidden_probe)  # 计算并保存当前步骤的中间状态。
assert before_forbid_path[1].action == "*2"  # 用受控断言验证关键不变量。
assert reflection == {"forbid": ((8, 1), "*2"), "reason": "overflow"}  # 用受控断言验证关键不变量。
assert after_forbid_path[1].action != "*2"  # 用受控断言验证关键不变量。

# forbidden 在 expansion 前已存在时，失败边甚至不会被重新创建。
fresh_root = Node(State(8, 1))  # 计算并保存当前步骤的中间状态。
expand(fresh_root, {state_key(fresh_root.state): fresh_root.stats}, forbidden_probe)  # 执行当前语句以推进本节示例。
assert "*2" not in fresh_root.children  # 用受控断言验证关键不变量。


## 8. 纯 simulation 与安全 commit：获批计划、一次性消费、逐步重新鉴权

规划函数只接收不可变 state，不接触真实环境。最终 plan 的每一步都来自一次新的 root visits/Q 决策；Approval 绑定起始快照、动作序列和预期中间状态。Commit 前校验摘要，只消费一次 approval，并在每一步读取当前 state、比对快照、重新调用 authorizer 后才执行副作用。


In [ ]:
def propose_plan(root_state, simulations=80, c=1.5):  # 定义本节可复用的核心函数。
    state, actions, decisions = root_state, [], []  # 计算并保存当前步骤的中间状态。
    for _ in range(MAX_STEPS - root_state.steps):  # 遍历输入元素以累积或检查结果。
        done, reward, _ = terminal_outcome(state)  # 计算并保存当前步骤的中间状态。
        if done:  # 按当前条件选择后续控制路径。
            if reward > 0:  # 按当前条件选择后续控制路径。
                return tuple(actions), tuple(decisions)  # 返回当前分支计算出的结果。
            raise RuntimeError("搜索到达失败终态")  # 遇到非法合同立即显式失败。
        tree, _, forbidden, _ = run_search(state, simulations, c)  # 计算并保存当前步骤的中间状态。
        action, child = choose_root_action(tree, forbidden)  # 计算并保存当前步骤的中间状态。
        decisions.append((state_key(state), action, child.visits, child.q))  # 计算并保存当前步骤的中间状态。
        actions.append(action)  # 计算并保存当前步骤的中间状态。
        state, _, _ = transition(state, action)  # 计算并保存当前步骤的中间状态。
    done, reward, _ = terminal_outcome(state)  # 计算并保存当前步骤的中间状态。
    if not done or reward <= 0:  # 按当前条件选择后续控制路径。
        raise RuntimeError("root 决策未在预算内形成成功计划")  # 遇到非法合同立即显式失败。
    return tuple(actions), tuple(decisions)  # 返回当前分支计算出的结果。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Approval:  # 定义承载本节状态与行为的数据结构。
    plan_id: str  # 执行当前语句以推进本节示例。
    start_state: State  # 执行当前语句以推进本节示例。
    actions: tuple  # 执行当前语句以推进本节示例。
    expected_states: tuple  # 执行当前语句以推进本节示例。

def approval_id(start_state, actions, expected_states):  # 定义本节可复用的核心函数。
    payload = {"start": asdict(start_state), "actions": actions, "expected": [asdict(state) for state in expected_states]}  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

def approve_successful_plan(start_state, actions):  # 定义本节可复用的核心函数。
    state, expected = start_state, []  # 计算并保存当前步骤的中间状态。
    for action in actions:  # 遍历输入元素以累积或检查结果。
        expected.append(state)  # 计算并保存当前步骤的中间状态。
        state, _, _ = transition(state, action)  # 计算并保存当前步骤的中间状态。
    done, reward, _ = terminal_outcome(state)  # 计算并保存当前步骤的中间状态。
    if not actions or not done or reward <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("只能批准可重放的成功计划")  # 遇到非法合同立即显式失败。
    expected = tuple(expected); actions = tuple(actions)  # 计算并保存当前步骤的中间状态。
    return Approval(approval_id(start_state, actions, expected), start_state, actions, expected)  # 返回当前分支计算出的结果。

@dataclass  # 为下方定义附加声明式配置。
class LiveEnvironment:  # 定义承载本节状态与行为的数据结构。
    state: State  # 执行当前语句以推进本节示例。
    applied_actions: list = field(default_factory=list)  # 计算并保存当前步骤的中间状态。
    approved_plan_ids: set = field(default_factory=set)  # 计算并保存当前步骤的中间状态。
    consumed_plan_ids: set = field(default_factory=set)  # 计算并保存当前步骤的中间状态。
    completed_plan_ids: set = field(default_factory=set)  # 计算并保存当前步骤的中间状态。

def commit_approved_plan(environment, approval, authorizer):  # 定义本节可复用的核心函数。
    if not isinstance(approval, Approval):  # 按当前条件选择后续控制路径。
        raise PermissionError("缺少结构化 Approval")  # 遇到非法合同立即显式失败。
    expected_id = approval_id(approval.start_state, approval.actions, approval.expected_states)  # 计算并保存当前步骤的中间状态。
    if approval.plan_id != expected_id:  # 按当前条件选择后续控制路径。
        raise PermissionError("Approval 摘要不匹配")  # 遇到非法合同立即显式失败。
    if approval.plan_id not in environment.approved_plan_ids:  # 按当前条件选择后续控制路径。
        raise PermissionError("计划未出现在真实批准集合")  # 遇到非法合同立即显式失败。
    if approval.plan_id in environment.consumed_plan_ids:  # 按当前条件选择后续控制路径。
        raise RuntimeError("Approval 已消费，禁止重复 commit")  # 遇到非法合同立即显式失败。
    if environment.state != approval.start_state:  # 按当前条件选择后续控制路径。
        raise RuntimeError("真实状态已偏离获批起点")  # 遇到非法合同立即显式失败。
    if not approval.actions or len(approval.actions) != len(approval.expected_states):  # 按当前条件选择后续控制路径。
        raise PermissionError("Approval 动作与预期状态长度不一致")  # 遇到非法合同立即显式失败。
    replay = approval.start_state  # 计算并保存当前步骤的中间状态。
    for action, expected_state in zip(approval.actions, approval.expected_states):  # 遍历输入元素以累积或检查结果。
        if replay != expected_state:  # 按当前条件选择后续控制路径。
            raise PermissionError("Approval 中间状态不可重放")  # 遇到非法合同立即显式失败。
        replay, _, _ = transition(replay, action)  # 计算并保存当前步骤的中间状态。
    if terminal_outcome(replay)[:2] != (True, 1.0):  # 按当前条件选择后续控制路径。
        raise PermissionError("Approval 未绑定成功终态")  # 遇到非法合同立即显式失败。
    environment.consumed_plan_ids.add(approval.plan_id)  # 计算并保存当前步骤的中间状态。
    for step, (action, expected_state) in enumerate(zip(approval.actions, approval.expected_states)):  # 遍历输入元素以累积或检查结果。
        observed = environment.state  # 计算并保存当前步骤的中间状态。
        if observed != expected_state:  # 按当前条件选择后续控制路径。
            raise RuntimeError("逐步观察发现状态漂移，停止执行")  # 遇到非法合同立即显式失败。
        if not authorizer(step, observed, action, approval):  # 按当前条件选择后续控制路径。
            raise PermissionError("当前步骤重新鉴权失败")  # 遇到非法合同立即显式失败。
        next_state, _, _ = transition(observed, action)  # 计算并保存当前步骤的中间状态。
        environment.state = next_state  # 计算并保存当前步骤的中间状态。
        environment.applied_actions.append(action)  # 计算并保存当前步骤的中间状态。
    environment.completed_plan_ids.add(approval.plan_id)  # 计算并保存当前步骤的中间状态。
    return environment.state  # 返回当前分支计算出的结果。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class LATSArtifact:  # 定义承载本节状态与行为的数据结构。
    environment: str  # 执行当前语句以推进本节示例。
    proposal: str  # 执行当前语句以推进本节示例。
    value_model: str  # 执行当前语句以推进本节示例。
    exploration_c: float  # 执行当前语句以推进本节示例。
    simulations: int  # 执行当前语句以推进本节示例。
    max_depth: int  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# simulation 只读 live snapshot；每一步 root 决策留下 visits/Q，而真实副作用仍为零。
live = LiveEnvironment(start)  # 计算并保存当前步骤的中间状态。
live_before = (live.state, tuple(live.applied_actions), set(live.approved_plan_ids), set(live.consumed_plan_ids))  # 计算并保存当前步骤的中间状态。
proposed_actions, root_decisions = propose_plan(live.state, simulations=80, c=1.5)  # 计算并保存当前步骤的中间状态。
assert (live.state, tuple(live.applied_actions), live.approved_plan_ids, live.consumed_plan_ids) == live_before  # 用受控断言验证关键不变量。
assert len(root_decisions) == len(proposed_actions) and all(visits > 0 and math.isfinite(q) for _, _, visits, q in root_decisions)  # 用受控断言验证关键不变量。

approval = approve_successful_plan(start, proposed_actions)  # 计算并保存当前步骤的中间状态。
unregistered_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    commit_approved_plan(live, approval, lambda *args: True)  # 执行当前语句以推进本节示例。
except PermissionError:  # 捕获预期异常并验证失败分支。
    unregistered_rejected = True  # 计算并保存当前步骤的中间状态。
assert unregistered_rejected and live.state == start and live.applied_actions == []  # 用受控断言验证关键不变量。

live.approved_plan_ids.add(approval.plan_id)  # 计算并保存当前步骤的中间状态。
tampered = Approval("bad-digest", approval.start_state, approval.actions, approval.expected_states)  # 计算并保存当前步骤的中间状态。
tampered_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    commit_approved_plan(live, tampered, lambda *args: True)  # 执行当前语句以推进本节示例。
except PermissionError:  # 捕获预期异常并验证失败分支。
    tampered_rejected = True  # 计算并保存当前步骤的中间状态。
assert tampered_rejected and live.state == start and live.applied_actions == []  # 用受控断言验证关键不变量。

# 合法 approval 每步重新鉴权一次；完成后再次提交被拒且没有重复副作用。
authorization_calls = []  # 计算并保存当前步骤的中间状态。
def step_authorizer(step, observed, action, approved):  # 定义本节可复用的核心函数。
    authorization_calls.append((step, observed, action, approved.plan_id))  # 计算并保存当前步骤的中间状态。
    return observed == approved.expected_states[step] and action == approved.actions[step]  # 返回当前分支计算出的结果。

committed_state = commit_approved_plan(live, approval, step_authorizer)  # 计算并保存当前步骤的中间状态。
applied_once = tuple(live.applied_actions)  # 计算并保存当前步骤的中间状态。
duplicate_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    commit_approved_plan(live, approval, step_authorizer)  # 执行当前语句以推进本节示例。
except RuntimeError:  # 捕获预期异常并验证失败分支。
    duplicate_rejected = True  # 计算并保存当前步骤的中间状态。
assert committed_state.value == TARGET and terminal_outcome(committed_state)[:2] == (True, 1.0)  # 用受控断言验证关键不变量。
assert len(authorization_calls) == len(approval.actions) == len(live.applied_actions)  # 用受控断言验证关键不变量。
assert all(call[1] == approval.expected_states[call[0]] for call in authorization_calls)  # 用受控断言验证关键不变量。
assert duplicate_rejected and tuple(live.applied_actions) == applied_once  # 用受控断言验证关键不变量。
assert approval.plan_id in live.completed_plan_ids  # 用受控断言验证关键不变量。

# 中途鉴权拒绝只执行此前已逐步获准的动作；approval 已消费，重试不会复制副作用。
denied_live = LiveEnvironment(start, approved_plan_ids={approval.plan_id})  # 计算并保存当前步骤的中间状态。
denied_calls = []  # 计算并保存当前步骤的中间状态。
def deny_second_step(step, observed, action, approved):  # 定义本节可复用的核心函数。
    denied_calls.append((step, observed, action))  # 计算并保存当前步骤的中间状态。
    return step == 0  # 返回当前分支计算出的结果。

midway_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    commit_approved_plan(denied_live, approval, deny_second_step)  # 执行当前语句以推进本节示例。
except PermissionError:  # 捕获预期异常并验证失败分支。
    midway_rejected = True  # 计算并保存当前步骤的中间状态。
denied_applied_once = tuple(denied_live.applied_actions)  # 计算并保存当前步骤的中间状态。
retry_rejected = False  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    commit_approved_plan(denied_live, approval, lambda *args: True)  # 执行当前语句以推进本节示例。
except RuntimeError:  # 捕获预期异常并验证失败分支。
    retry_rejected = True  # 计算并保存当前步骤的中间状态。
assert midway_rejected and len(denied_calls) == 2 and len(denied_applied_once) == 1  # 用受控断言验证关键不变量。
assert retry_rejected and tuple(denied_live.applied_actions) == denied_applied_once  # 用受控断言验证关键不变量。
assert approval.plan_id in denied_live.consumed_plan_ids and approval.plan_id not in denied_live.completed_plan_ids  # 用受控断言验证关键不变量。

# 制品绑定实际搜索常数与预算；模拟轨迹仍显式标记 simulated。
simulated_actions = [{"action": action, "mode": "simulated"} for action in proposed_actions]  # 计算并保存当前步骤的中间状态。
artifact = LATSArtifact("number-env-v2", "root-visits-q-v2", "distance-v1", 1.5, 80, MAX_STEPS)  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert all(item["mode"] == "simulated" for item in simulated_actions)  # 用受控断言验证关键不变量。
assert artifact.simulations > 0 and artifact.max_depth == MAX_STEPS  # 用受控断言验证关键不变量。
assert digest != artifact_hash(LATSArtifact("number-env-v3", artifact.proposal, artifact.value_model, 1.5, 80, MAX_STEPS))  # 用受控断言验证关键不变量。


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
